# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasmeenmh90-beep/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import os, sys, subprocess
REPO_URL = "https://github.com/yasmeenmh90-beep/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

%pip -q install duckdb huggingface_hub
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'
fact_month = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
print("Connected. Working dir:", os.getcwd())

Connected. Working dir: /content/flyrank-ml-internship-starter


In [2]:
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

SEED = 42

print(
    "Versions — numpy", np.__version__,
    "| pandas", pd.__version__,
    "| scikit-learn", sklearn.__version__,
    "| seed", SEED
)

Versions — numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1 | seed 42


# 1. Two paper findings + my methodology questions

The purpose of this section is not to grade the paper. I am using two of its
findings to practise asking the same validation questions that I should ask
about my own model.

## Finding 1

**Finding reported in the paper:**  
[PASTE THE FIRST EXACT FINDING FROM THE PAPER HERE.]

**Methodology question:**  
How was the outcome used for this finding defined, and was all information
used to construct the predictors available before that outcome occurred?

This matters because a strong measured relationship can be misleading if the
label definition overlaps with information available to the model. I would
want to understand the timing of the features, the timing of the outcome, and
whether any part of the outcome could indirectly enter the predictors.

## Finding 2

**Finding reported in the paper:**  
[PASTE THE SECOND EXACT FINDING FROM THE PAPER HERE.]

**Methodology question:**  
Does the validation design test generalisation to genuinely unseen units, or
could related observations appear in both training and evaluation data?

For example, if several observations belong to the same client, a random
row-level split can allow one client's patterns to occur on both sides of the
split. A grouped or time-aware evaluation would provide stronger evidence for
a claim about performance on unseen clients or future observations.

These questions do not imply that either finding is incorrect. They identify
the additional methodological information I would want before interpreting
the findings more broadly.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

raw = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        COALESCE(
            SUM(CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_impressions
            END), 0
        ) AS imp_first_half,

        COALESCE(
            SUM(CASE
                WHEN report_date > DATE '2026-03-15'
                THEN gsc_impressions
            END), 0
        ) AS imp_second_half,

        COALESCE(
            SUM(CASE
                WHEN report_date <= DATE '2026-03-07'
                THEN gsc_impressions
            END), 0
        ) AS imp_days_1_7,

        COALESCE(
            SUM(CASE
                WHEN report_date > DATE '2026-03-07'
                 AND report_date <= DATE '2026-03-15'
                THEN gsc_impressions
            END), 0
        ) AS imp_days_8_15,

        COALESCE(
            SUM(CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_clicks
            END), 0
        ) AS clicks_first_half,

        AVG(CASE
            WHEN report_date <= DATE '2026-03-15'
            THEN gsc_avg_position
        END) AS pos_first_half,

        COALESCE(
            SUM(CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN ga4_pageviews
            END), 0
        ) AS pageviews_first_half,

        COALESCE(
            SUM(CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN ga4_engaged_sessions
            END), 0
        ) AS engaged_first_half,

        BOOL_OR(ga4_data_available) AS has_ga4

    FROM {fact_month}
    GROUP BY 1, 2
    HAVING imp_first_half >= 50
    ORDER BY client_hash_id, content_hash_id
""").df()

df = raw.copy()

df["has_ga4"] = df["has_ga4"].fillna(False).astype(bool)

num_cols = [
    "imp_first_half",
    "imp_second_half",
    "imp_days_1_7",
    "imp_days_8_15",
    "clicks_first_half",
    "pos_first_half",
    "pageviews_first_half",
    "engaged_first_half",
]

for c in num_cols:
    df[c] = df[c].astype(float)

# Same Week-5 target
df["is_declining"] = (
    df["imp_second_half"] < 0.8 * df["imp_first_half"]
).astype(int)

# Same Week-5 engineered features
df["log_imp_first_half"] = np.log1p(df["imp_first_half"])

df["ctr_first_half"] = (
    df["clicks_first_half"] /
    df["imp_first_half"]
)

df["momentum_first_half"] = np.log(
    ((df["imp_days_8_15"] / 8) + 1) /
    ((df["imp_days_1_7"] / 7) + 1)
)

df["log_pageviews_first_half"] = np.log1p(
    df["pageviews_first_half"]
)

df["engagement_rate"] = (
    df["engaged_first_half"] /
    df["pageviews_first_half"].replace(0, np.nan)
)

df["has_ga4_int"] = df["has_ga4"].astype(int)

FEATURES = [
    "log_imp_first_half",
    "pos_first_half",
    "ctr_first_half",
    "momentum_first_half",
    "log_pageviews_first_half",
    "engagement_rate",
    "has_ga4_int",
]

TARGET = "is_declining"
GROUP = "client_hash_id"

X = df[FEATURES]
y = df[TARGET]
groups = df[GROUP]

print("Rows:", len(df))
print("Clients:", groups.nunique())
print("Decline base rate:", round(y.mean(), 3))

display(df[FEATURES + [TARGET]].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 92548
Clients: 40
Decline base rate: 0.286


,log_imp_first_half,pos_first_half,ctr_first_half,momentum_first_half,log_pageviews_first_half,engagement_rate,has_ga4_int,is_declining
0,4.787492,12.639599,0.008403,0.106419,0.0,NaN,0,0
1,4.290459,8.094074,0.000000,0.074108,0.0,NaN,0,0
2,5.648974,12.587226,0.000000,-0.079827,0.0,NaN,0,1
3,4.204693,12.282875,0.000000,-0.257145,0.0,NaN,0,0
4,6.089045,11.741716,0.004545,-0.076373,0.0,NaN,0,1


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def make_rf():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=50,
            random_state=SEED,
            n_jobs=-1
        )
    )


def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores, kind="stable")
    k = min(k, len(order))

    return labels[order[:k]].mean()


def evaluate_split(name, train_idx, test_idx):
    model = make_rf()

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]

    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])

    return {
        "split": name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_clients": len(train_clients),
        "test_clients": len(test_clients),
        "client_overlap": len(train_clients & test_clients),
        "test_base_rate": y_test.mean(),
        "AUC": roc_auc_score(y_test, scores),
        "avg_precision": average_precision_score(y_test, scores),
        "P@500": precision_at_k(scores, y_test.values, 500),
    }, model, scores

In [5]:
indices = np.arange(len(df))

random_train_idx, random_test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=SEED,
    stratify=y
)

before_result, before_model, before_scores = evaluate_split(
    "Before: random row split",
    random_train_idx,
    random_test_idx
)

pd.DataFrame([before_result]).round(3)

,split,train_rows,test_rows,train_clients,test_clients,client_overlap,test_base_rate,AUC,avg_precision,P@500
0,Before: random row split,69411,23137,40,36,36,0.286,0.734,0.545,0.83


In [6]:
random_train_clients = set(groups.iloc[random_train_idx])
random_test_clients = set(groups.iloc[random_test_idx])

random_overlap = random_train_clients & random_test_clients

print("Train clients:", len(random_train_clients))
print("Test clients:", len(random_test_clients))
print("Clients appearing in BOTH:", len(random_overlap))

Train clients: 40
Test clients: 36
Clients appearing in BOTH: 36


In [7]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED
)

group_train_idx, group_test_idx = next(
    gss.split(X, y, groups=groups)
)

after_result, after_model, after_scores = evaluate_split(
    "After: GroupShuffleSplit by client",
    group_train_idx,
    group_test_idx
)

pd.DataFrame([after_result]).round(3)

,split,train_rows,test_rows,train_clients,test_clients,client_overlap,test_base_rate,AUC,avg_precision,P@500
0,After: GroupShuffleSplit by client,67662,24886,30,10,0,0.369,0.738,0.632,0.864


In [8]:
group_train_clients = set(groups.iloc[group_train_idx])
group_test_clients = set(groups.iloc[group_test_idx])

overlap = group_train_clients & group_test_clients

print("Train clients:", len(group_train_clients))
print("Test clients:", len(group_test_clients))
print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("PASS: grouped evaluation contains no client overlap.")

Train clients: 30
Test clients: 10
Client overlap: 0
PASS: grouped evaluation contains no client overlap.


In [9]:
comparison = pd.DataFrame([
    before_result,
    after_result
]).set_index("split")

display(
    comparison[
        [
            "train_rows",
            "test_rows",
            "train_clients",
            "test_clients",
            "client_overlap",
            "test_base_rate",
            "AUC",
            "avg_precision",
            "P@500",
        ]
    ].round(3)
)

,train_rows,test_rows,train_clients,test_clients,client_overlap,test_base_rate,AUC,avg_precision,P@500
split,,,,,,,,,
Before: random row split,69411,23137,40,36,36,0.286,0.734,0.545,0.830
After: GroupShuffleSplit by client,67662,24886,30,10,0,0.369,0.738,0.632,0.864


### Interpretation

The random row split is the weaker validation design because observations from
the same client can appear in both training and test data. Therefore, its
metrics measure performance on new content rows drawn partly from clients the
model has already encountered.

The grouped split changes the question. All rows belonging to a held-out
client remain outside training, so the measured performance is evidence about
generalisation to the held-out clients in this dataset.

I therefore treat the grouped result as the more appropriate estimate for an
unseen-client use case. I do not interpret either result as evidence of causal
effects or guaranteed future performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

audit_rows = [
    {
        "feature": "log_imp_first_half",
        "source_window": "days 1-15",
        "available_at_decision_time": True,
        "label_information": False,
        "assessment": "OK"
    },
    {
        "feature": "pos_first_half",
        "source_window": "days 1-15",
        "available_at_decision_time": True,
        "label_information": False,
        "assessment": "OK"
    },
    {
        "feature": "ctr_first_half",
        "source_window": "days 1-15",
        "available_at_decision_time": True,
        "label_information": False,
        "assessment": "OK"
    },
    {
        "feature": "momentum_first_half",
        "source_window": "days 1-15 only",
        "available_at_decision_time": True,
        "label_information": False,
        "assessment": "OK, but closely related to target construction"
    },
    {
        "feature": "log_pageviews_first_half",
        "source_window": "days 1-15",
        "available_at_decision_time": True,
        "label_information": False,
        "assessment": "OK"
    },
    {
        "feature": "engagement_rate",
        "source_window": "days 1-15",
        "available_at_decision_time": True,
        "label_information": False,
        "assessment": "OK"
    },
    {
        "feature": "has_ga4_int",
        "source_window": "availability indicator",
        "available_at_decision_time": True,
        "label_information": False,
        "assessment": "Review availability semantics"
    }
]

leakage_audit = pd.DataFrame(audit_rows)

display(leakage_audit)

,feature,source_window,available_at_decision_time,label_information,assessment
0,log_imp_first_half,days 1-15,True,False,OK
1,pos_first_half,days 1-15,True,False,OK
2,ctr_first_half,days 1-15,True,False,OK
3,momentum_first_half,days 1-15 only,True,False,"OK, but closely related to target construction"
4,log_pageviews_first_half,days 1-15,True,False,OK
5,engagement_rate,days 1-15,True,False,OK
6,has_ga4_int,availability indicator,True,False,Review availability semantics


In [11]:
# No feature should explicitly contain second-half information.
assert not any(
    "second" in feature.lower()
    for feature in FEATURES
)

# Target itself must never be a predictor.
assert TARGET not in FEATURES

# Identifiers must not be model features.
assert "client_hash_id" not in FEATURES
assert "content_hash_id" not in FEATURES

print("PASS: no explicit target, identifier, or second-half feature is used.")

PASS: no explicit target, identifier, or second-half feature is used.


In [12]:
group_test = df.iloc[group_test_idx].copy()

leaky_pct = (
    group_test["imp_second_half"] -
    group_test["imp_first_half"]
) / group_test["imp_first_half"]

leaky_score = np.where(
    leaky_pct < -0.30,
    np.abs(leaky_pct) * np.log1p(group_test["imp_first_half"]),
    0.0
)

leaky_auc = roc_auc_score(
    group_test[TARGET],
    leaky_score
)

honest_auc = roc_auc_score(
    group_test[TARGET],
    after_scores
)

print("Honest Random Forest AUC:", round(honest_auc, 3))
print("Leaky second-half reference AUC:", round(leaky_auc, 3))

Honest Random Forest AUC: 0.738
Leaky second-half reference AUC: 0.911


### Leakage interpretation

The target is defined using second-half impressions. Therefore,
`imp_second_half`, or any feature calculated from it, is unavailable at the
day-15 decision point and would leak outcome information.

The intentionally leaky reference illustrates the effect. Its stronger metric
does not represent a better deployable model because it uses information from
the period the model is supposed to predict.

The production-style feature set contains only information available by day
15. `momentum_first_half` deserves particular caution because it is closely
related to the target mathematically: it measures the recent first-half
trajectory. It is available at prediction time, so I do not classify it as
temporal leakage, but I interpret it as a strong recent-trend signal rather
than evidence of a causal driver.

In [13]:
availability_check = con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS n,
        SUM(
            CASE WHEN gsc_impressions IS NULL
            THEN 1 ELSE 0 END
        ) AS null_impressions,
        SUM(
            CASE WHEN gsc_impressions = 0
            THEN 1 ELSE 0 END
        ) AS zero_impressions
    FROM {fact_month}
    GROUP BY 1
    ORDER BY 1
""").df()

display(availability_check)

,gsc_data_available,n,null_impressions,zero_impressions
0,False,6230317,0.0,6230317.0
1,True,3611061,0.0,0.0


### Label-quality limitation

The audit also identified a data-quality limitation. Rows where Search Console
data are unavailable can contain zero impressions. This means a zero can
represent either genuinely zero observed impressions or unavailable data,
depending on the warehouse convention.

Because the decline label compares first-half and second-half impression
totals, missing coverage can affect the measured label. I therefore treat
`is_declining` as an operational label derived from the available warehouse
data rather than as perfect ground truth.

In [14]:
audit_test = df.iloc[group_test_idx].copy()

audit_test["p_decline"] = after_scores

audit_test["predicted_class"] = (
    audit_test["p_decline"] >= 0.5
).astype(int)

audit_test["error_type"] = np.select(
    [
        (audit_test[TARGET] == 0) &
        (audit_test["predicted_class"] == 1),

        (audit_test[TARGET] == 1) &
        (audit_test["predicted_class"] == 0),
    ],
    [
        "False positive",
        "False negative",
    ],
    default="Correct"
)

print("Error counts:")
display(audit_test["error_type"].value_counts().to_frame())

Error counts:


,count
error_type,
Correct,17284
False negative,6838
False positive,764


In [15]:
false_positives = (
    audit_test[
        (audit_test[TARGET] == 0) &
        (audit_test["predicted_class"] == 1)
    ]
    .sort_values("p_decline", ascending=False)
    .head(5)
)

display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "p_decline",
            "imp_first_half",
            "imp_second_half",
            "momentum_first_half",
            "ctr_first_half",
            "pos_first_half",
            "engagement_rate",
        ]
    ].round(3)
)

,client_hash_id,content_hash_id,p_decline,imp_first_half,imp_second_half,momentum_first_half,ctr_first_half,pos_first_half,engagement_rate
26935,client_3f0ce4d44fe94f3d,content_72e44a5843ee12bd,0.811,63.0,530.0,-2.303,0.000,7.403,NaN
43744,client_62f4a7e64f5e0096,content_da0893d91badaac5,0.803,1130.0,1550.0,-2.375,0.001,6.728,NaN
71350,client_a80fca3f171ed1de,content_fb08358374ad748f,0.800,190.0,194.0,-2.108,0.000,6.678,NaN
42076,client_62f4a7e64f5e0096,content_c0ca95dac28acf05,0.799,89.0,72.0,-1.838,0.000,5.633,NaN
6500,client_0fa64a184f18a4a0,content_90d752058c3f503e,0.792,328.0,377.0,-1.999,0.000,2.420,NaN


In [16]:
false_negatives = (
    audit_test[
        (audit_test[TARGET] == 1) &
        (audit_test["predicted_class"] == 0)
    ]
    .sort_values("p_decline", ascending=True)
    .head(5)
)

display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "p_decline",
            "imp_first_half",
            "imp_second_half",
            "momentum_first_half",
            "ctr_first_half",
            "pos_first_half",
            "engagement_rate",
        ]
    ].round(3)
)

,client_hash_id,content_hash_id,p_decline,imp_first_half,imp_second_half,momentum_first_half,ctr_first_half,pos_first_half,engagement_rate
6734,client_0fa64a184f18a4a0,content_f02ed5becdd01388,0.030,1004.0,572.0,4.840,0.013,1.440,0.0
6419,client_0fa64a184f18a4a0,content_6c8850d1588d3eb7,0.032,156.0,16.0,3.020,0.019,3.216,0.0
6286,client_0fa64a184f18a4a0,content_325b0949225ed278,0.033,1829.0,88.0,0.307,0.008,2.127,0.0
6650,client_0fa64a184f18a4a0,content_d5e927886fbdd9f1,0.035,100.0,49.0,2.603,0.010,2.319,0.0
6283,client_0fa64a184f18a4a0,content_30c7f0bbdb7e74c9,0.035,372.0,207.0,1.920,0.008,3.777,0.0


In [17]:
cm = confusion_matrix(
    audit_test[TARGET],
    audit_test["predicted_class"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual: no decline", "Actual: decline"],
    columns=["Predicted: no decline", "Predicted: decline"]
)

display(cm_df)

,Predicted: no decline,Predicted: decline
Actual: no decline,14950,764
Actual: decline,6838,2334


### Error interpretation

The failure examples show that the model should be treated as a ranking or
decision-support signal rather than an automatic decision rule.

A high predicted probability does not guarantee that an item will decline,
and some genuinely declining items receive low scores. These errors are
especially relevant because the model relies heavily on recent first-half
momentum.

The examples also help separate predictive association from explanation. A
feature such as falling recent momentum can help rank items while still not
telling us why the decline occurs.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 4. Claim rewrite

## Claim I would avoid

> The Random Forest accurately predicts which pages will decline and proves
> that momentum causes future traffic loss.

This wording goes beyond the evidence. The experiment is observational, the
label is constructed from warehouse measurements, performance varies across
clients, and the strongest feature is closely related to the outcome
definition.

## Revised public-safe claim

> In the March 2026 evaluation, a Random Forest using only information
> available through day 15 showed directional ability to rank content by the
> observed likelihood of a lower second-half impression total. Under a
> client-grouped holdout, the model measured an AUC of approximately 0.74 and
> P@500 of approximately 0.86. Grouped cross-validation produced a lower mean
> AUC of approximately 0.69, indicating variation across client groups.
> Recent first-half impression momentum carried most of the predictive signal.
> These results support use as a prioritisation or decision-support signal,
> not as evidence that the features cause traffic decline or that the same
> performance will hold for every client or future month.

## Short stakeholder version

> In this dataset, first-half signals — especially recent impression momentum —
> were associated with second-half decline. The model can help prioritise
> content for review, but its scores should be treated as decision support
> rather than guaranteed forecasts.

## What the evidence supports

The evaluation supports statements about observed predictive performance on
the held-out clients in this dataset. It does not establish causality,
universal performance across clients, or future production performance.

## Self-check



In [18]:
checks = {
    "Target is not a feature":
        TARGET not in FEATURES,

    "Client ID is not a feature":
        "client_hash_id" not in FEATURES,

    "Content ID is not a feature":
        "content_hash_id" not in FEATURES,

    "No explicit second-half feature":
        not any("second" in f.lower() for f in FEATURES),

    "Grouped split has zero client overlap":
        len(
            set(groups.iloc[group_train_idx]) &
            set(groups.iloc[group_test_idx])
        ) == 0,

    "Model evaluation has test rows":
        len(group_test_idx) > 0,

    "Failure examples were inspected":
        (
            len(false_positives) > 0 or
            len(false_negatives) > 0
        ),
}

self_check = pd.DataFrame(
    {
        "check": list(checks.keys()),
        "passed": list(checks.values())
    }
)

display(self_check)

assert all(checks.values())

print("All validation-audit checks passed.")

,check,passed
0,Target is not a feature,True
1,Client ID is not a feature,True
2,Content ID is not a feature,True
3,No explicit second-half feature,True
4,Grouped split has zero client overlap,True
5,Model evaluation has test rows,True
6,Failure examples were inspected,True


All validation-audit checks passed.


# Final self-check

- [x] I discussed two research findings and asked a concrete methodology
      question about each.
- [x] I compared a random row split with a client-grouped split.
- [x] The grouped evaluation contains no client overlap.
- [x] I audited every model feature for timing and leakage.
- [x] I demonstrated why using second-half information would be leakage.
- [x] I inspected concrete false-positive and false-negative examples.
- [x] I documented a limitation in the decline label caused by data coverage.
- [x] I rewrote the model claim using observed/measured language.
- [x] I describe the model as decision support rather than an automatic
      decision system.
- [x] I do not make causal claims from this observational evaluation.

## Final conclusion

The audit changes how I interpret the Week-5 result more than it changes the
model itself. The model contains useful directional signal, but validation
design, client concentration, missing-data conventions, and the dominance of
the momentum feature all limit how broadly the measured metrics should be
generalised.

The client-grouped evaluation is the more relevant test for an unseen-client
use case. The results support using the score to prioritise content for human
review. They do not establish that the model identifies the cause of decline
or that its measured performance will automatically transfer to new clients,
months, or production conditions.